# Your first Claude tool — search PubMed conversationally

**Companion notebook** for the rendered note at
👉 [tensoromics.com/notes/claude-tools-getting-started](https://tensoromics.com/notes/claude-tools-getting-started)

What you'll build:

1. A 6-line Python function `pubmed_count()` that hits NCBI's E-utilities and returns the number of PubMed papers matching a query.
2. A second function `pubmed_search()` that returns title, year, journal, and authors for the N most recent hits.
3. Both functions wired into the Anthropic Claude API as **tools**, so you can ask literature questions in plain English and Claude calls the right function for you.
4. Three worked examples: a direct count call, a conversational count via Claude, and a chained call where Claude uses both tools to answer one question.

> **Editing workflow.** Code in this notebook is mirrored byte-for-byte in `tensoromics-web/content/notes/claude-tools-getting-started.mdx`. When you change one, port the change to the other.

## 0. Setup

In [ ]:
%pip install --quiet requests anthropic

In [ ]:
import os
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."  # or export in shell before launching Jupyter
# (only needed for Section 3 below; Sections 1 and 2 work without an API key)

## 1. The tools

NCBI's [E-utilities](https://www.ncbi.nlm.nih.gov/books/NBK25497/) is a free, no-key API that wraps PubMed, GenBank, GEO, and most other NCBI databases. We use two endpoints:

- `esearch` — returns matching record IDs (PMIDs) and a total count.
- `esummary` — returns JSON metadata for a list of IDs.

In [ ]:
import requests

EUTILS = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"

def pubmed_count(query: str, year: int | None = None) -> int:
    """Count PubMed papers matching `query`. Optionally restrict to a year."""
    term = f"{query} AND {year}[pdat]" if year else query
    r = requests.get(
        f"{EUTILS}/esearch.fcgi",
        params={"db": "pubmed", "term": term, "rettype": "count", "retmode": "json"},
        timeout=10,
    )
    r.raise_for_status()
    return int(r.json()["esearchresult"]["count"])

def pubmed_search(query: str, n: int = 5) -> list[dict]:
    """Return up to `n` most recent PubMed papers matching `query`."""
    # 1. esearch — PMIDs sorted by publication date (newest first).
    r = requests.get(
        f"{EUTILS}/esearch.fcgi",
        params={"db": "pubmed", "term": query, "sort": "pub_date",
                "retmax": n, "retmode": "json"},
        timeout=10,
    )
    r.raise_for_status()
    pmids = r.json()["esearchresult"]["idlist"]
    if not pmids:
        return []

    # 2. esummary — JSON metadata for those PMIDs.
    r = requests.get(
        f"{EUTILS}/esummary.fcgi",
        params={"db": "pubmed", "id": ",".join(pmids), "retmode": "json"},
        timeout=10,
    )
    r.raise_for_status()
    docs = r.json()["result"]

    return [{
        "pmid": pmid,
        "title": docs[pmid].get("title", ""),
        "year": docs[pmid].get("pubdate", "")[:4],
        "journal": docs[pmid].get("source", ""),
        "authors": [a["name"] for a in docs[pmid].get("authors", [])[:3]],
    } for pmid in pmids]

## 2. Worked example 1 — count papers, called directly

How crowded is the KRAS G12C inhibitor field in 2026?

In [ ]:
n = pubmed_count("KRAS G12C inhibitor", year=2026)
print(f"PubMed papers on KRAS G12C inhibitors in 2026: {n}")

**Expected** (count varies as PubMed indexes new papers): a single integer line like `PubMed papers on KRAS G12C inhibitors in 2026: 487`. One HTTP call, one number.

## 3. Wire the tools to Claude (needs `ANTHROPIC_API_KEY`)

Two pieces: a small **declaration** that tells Claude both tools exist, and a short **loop** that runs the conversation. The loop dispatches to the right function based on `tu.name`.

In [ ]:
assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY before continuing."

### 3a. Declare the tools

A tool declaration is a small dict that tells Claude what the function is called, when to use it, and what arguments it takes. We start with imports and client setup, then declare the first tool.

In [ ]:
import json
from anthropic import Anthropic

client = Anthropic()  # reads ANTHROPIC_API_KEY from the environment
MODEL = "claude-opus-4-7"  # sonnet / haiku also work and cost less

count_tool = {
    "name": "pubmed_count",
    "description": (
        "Count PubMed papers matching a query, optionally restricted to a "
        "publication year. Use when the user asks how many papers exist "
        "on a topic."
    ),
    "input_schema": {
        "type": "object",
        "required": ["query"],
        "properties": {
            "query": {"type": "string"},
            "year": {"type": "integer"},
        },
    },
}

**Anatomy of a tool declaration:**

- `name` — the exact function name. Claude uses this to dispatch back to your code when it returns a `tool_use` block.
- `description` — the single most important field. Claude reads it to decide *whether* (and *which*) tool to call. Be specific about *when* it applies, not just what it does.
- `input_schema.required` — argument names Claude must provide.
- `input_schema.properties` — JSON Schema for each argument (`"type": "string" | "integer" | "boolean" | "array"`, etc.).

The second tool follows the exact same shape. We collect both into a `TOOLS` list, and keep a parallel `TOOL_FUNCS` dict so the loop in 3b can call the right function by name.

In [ ]:
search_tool = {
    "name": "pubmed_search",
    "description": (
        "Return the N most recent PubMed papers matching a query, with "
        "title, year, journal, and first three authors. Use when the user "
        "asks for recent papers, latest research, or a list of papers on a topic."
    ),
    "input_schema": {
        "type": "object",
        "required": ["query"],
        "properties": {
            "query": {"type": "string"},
            "n": {"type": "integer"},
        },
    },
}

TOOLS = [count_tool, search_tool]
TOOL_FUNCS = {"pubmed_count": pubmed_count, "pubmed_search": pubmed_search}

### 3b. Run the conversation loop

Three steps, repeated until Claude is done: send the question, execute any tool call Claude requests, send the result back.

In [ ]:
def chat_with_tools(question: str) -> str:
    messages = [{"role": "user", "content": question}]
    while True:
        resp = client.messages.create(
            model=MODEL, max_tokens=2048, tools=TOOLS, messages=messages,
        )
        if resp.stop_reason == "end_turn":
            return next(b.text for b in resp.content if b.type == "text")

        tu = next(b for b in resp.content if b.type == "tool_use")
        result = TOOL_FUNCS[tu.name](**tu.input)
        messages += [
            {"role": "assistant", "content": resp.content},
            {"role": "user", "content": [{
                "type": "tool_result",
                "tool_use_id": tu.id,
                "content": json.dumps(result),
            }]},
        ]

## 4. Worked example 2 — ask Claude conversationally

In [ ]:
print(chat_with_tools("How many papers on KRAS G12C inhibitors were published in 2026?"))

**Expected** (wording will vary): a prose answer that includes the count (e.g., 487) and likely a short comment about the field's activity.

Behind the scenes Claude inspected the question, matched it against `pubmed_count`'s description, called `pubmed_count(query="KRAS G12C inhibitor", year=2026)`, read the result, and wrote the answer. The number came from PubMed at request time, not from Claude's training memory.

## 5. Worked example 3 — chained calls

A real research question often needs both tools. Claude is good at deciding the sequence.

In [ ]:
print(chat_with_tools(
    "How active is spatial transcriptomics as a field — how many papers "
    "in 2026 so far, and show me the 3 most recent ones?"
))

**Expected** (PMIDs and titles will differ): a prose answer that opens with the count of 2026 spatial transcriptomics papers, followed by 3 recent titles with year, journal, and first authors.

Two tool calls, one prose answer. Claude called `pubmed_count` first, then `pubmed_search` with `n=3`, then wove both results into a single reply.

## 6. Where to take this next

- **Add a third tool: `pubmed_abstract(pmid)`.** Same pattern, but calls `efetch` with `rettype=abstract`. Now Claude can read and summarise specific papers.
- **Search bioRxiv too.** Register a `biorxiv_recent(topic, days=30)` tool alongside the PubMed ones. Claude will pick the right one based on whether the user asks for "preprints" or "papers".
- **Cache results.** Wrap `pubmed_count` and `pubmed_search` in `functools.lru_cache` or a disk cache. The literature changes daily, but identical queries in the same session don't need to re-hit NCBI.
- **Move to richer tools.** Once the basic shape is comfortable, the [gene ID resolver note](https://tensoromics.com/notes/gene-id-resolver/) shows the same pattern with a more involved function (MyGene response handling, DataFrame enrichment, deprecated-alias detection).